# Task 3 — Define Input Schema and Validate

This notebook defines the common input schema and validates the cleaned PNU dataset against the required rules.

The validation checks include:
- Required columns
- Data structure
- Allowed values
- Duplicate handling
- Required field validation

Valid records will be saved as validated.csv, while failed records will be saved as rejected.csv with validation reasons.

In [1]:
# Import required libraries for data validation

import pandas as pd
from pathlib import Path

In [2]:
# Define project directories

project_root = Path("..")

interim_dir = project_root / "data" / "interim"

In [3]:
# Load validated dataset generated from Task 2

input_path = interim_dir / "PNU_validated.csv"

df = pd.read_csv(input_path)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Dataset loaded successfully
Rows: 1544
Columns: 13


,research_id,university,title,authors,publication_year,publication_date,abstract,research_field,tech_category,journal,doi,url,source
0,PNU-0,PNU,sexual workplace violence in the health sector...,Aseel Khaled AlHassan; Reem Tarik AlSaqat; Fah...,2023,2023-10-06,<jats:title>Abstract</jats:title>\n ...,humanitarian,NaN,BMC Health Services Research,10.1186/s12913-023-10080-y,https://doi.org/10.1186/s12913-023-10080-y,PNU Open Data
1,PNU-2,PNU,"association of ccnd1 (c.723g > a, rs9344) vari...",Mohamed Adel El-Eshmawy; Hanaa Elsayed Shahin;...,2023,NaN,NaN,humanitarian,NaN,Molecular Biology Reports,10.1007/s11033-022-08202-6,https://doi.org/10.1007/s11033-022-08202-6,PNU Open Data
2,PNU-3,PNU,neuroprotective effect of barbaloin on strepto...,Asma B. Omer; Obaid Afzal; Abdulmalik S. A. Al...,2023,2023-02-28,NaN,humanitarian,NaN,ACS Omega,10.1021/acsomega.2c08277,https://doi.org/10.1021/acsomega.2c08277,PNU Open Data
3,PNU-4,PNU,adoption of antenatal care conversation mappin...,Anwar Alhashem; Bayader A. Alotaiby; Rahaf B. ...,2023,2023-06-08,"<jats:sec id=""sec001"">\n<jats:title>Aim</jats:...",humanitarian,NaN,PLOS ONE,10.1371/journal.pone.0286656,https://doi.org/10.1371/journal.pone.0286656,PNU Open Data
4,PNU-12,PNU,the role of digital resources in enhancing the...,Mohamed Abdelfattah; Heba Abdel-Aziz Abu-Bakr;...,2023,2023-06-14,<jats:p>The quality and quantity of groundwate...,health-related / medical,NaN,Frontiers in Earth Science,10.3389/feart.2023.1204742,https://doi.org/10.3389/feart.2023.1204742,PNU Open Data


# Define the common schema required for all university datasets

In [4]:
# Define the common schema based on the README structure

schema = pd.DataFrame([
    ["research_id", "string", False, "Non-empty and unique"],
    ["university", "string", False, "PNU"],
    ["title", "string", False, "Non-empty text"],
    ["authors", "string", False, "Required author information"],
    ["publication_year", "integer", False, "Valid publication year"],
    ["publication_date", "date", True, "Valid date if available"],
    ["abstract", "string", True, "Text if available"],
    ["research_field", "string", True, "Text if available"],
    ["tech_category", "string", True, "Text if available"],
    ["journal", "string", True, "Text if available"],
    ["doi", "string", False, "Required DOI"],
    ["url", "string", False, "Required valid URL"],
    ["source", "string", False, "Dataset source"]
],
columns=[
    "column",
    "expected_type",
    "nullable",
    "allowed_values_or_range"
])

schema

,column,expected_type,nullable,allowed_values_or_range
0,research_id,string,False,Non-empty and unique
1,university,string,False,PNU
2,title,string,False,Non-empty text
3,authors,string,False,Required author information
4,publication_year,integer,False,Valid publication year
5,publication_date,date,True,Valid date if available
6,abstract,string,True,Text if available
7,research_field,string,True,Text if available
8,tech_category,string,True,Text if available
9,journal,string,True,Text if available


In [5]:
# Check if dataset columns match the common schema

expected_columns = schema["column"].tolist()

print("PNU rows:", len(df))
print("PNU columns:", len(df.columns))

print(
    "Columns match schema:",
    df.columns.tolist() == expected_columns
)

print(
    "Duplicate research IDs:",
    df["research_id"].duplicated().sum()
)

PNU rows: 1544
PNU columns: 13
Columns match schema: True
Duplicate research IDs: 0


In [6]:
# Function to validate PNU dataset against the common schema rules

def validate_dataset(df):

    # Create a copy to keep the original dataset unchanged
    result = df.copy()

    # Create a column to store validation errors
    errors = pd.Series(
        "",
        index=result.index,
        dtype="string"
    )


    # Check required columns based on schema
    required_columns = schema.loc[
        schema["nullable"] == False,
        "column"
    ].tolist()


    for column in required_columns:

        missing = (
            result[column].isna()
            |
            result[column].astype("string").str.strip().eq("")
        )

        errors.loc[missing] += f"{column} is missing; "


    # Check duplicate research IDs

    duplicate_ids = result["research_id"].duplicated(
        keep=False
    )

    errors.loc[duplicate_ids] += "duplicate research_id; "


    # Validate publication year

    years = pd.to_numeric(
        result["publication_year"],
        errors="coerce"
    )

    invalid_year = years.isna()

    errors.loc[invalid_year] += "invalid publication year; "


    # Validate URL format only when URL exists

    url_present = result["url"].notna()

    invalid_url = (
        url_present
        &
        ~result["url"].astype("string").str.match(
            r"^https?://",
            na=False
        )
    )

    errors.loc[invalid_url] += "invalid URL; "


    # Add validation result column

    result["validation_error"] = (
        errors.str.rstrip("; ")
    )


    # Separate valid and rejected records

    validated = result[
        result["validation_error"] == ""
    ].copy()


    rejected = result[
        result["validation_error"] != ""
    ].copy()


    return validated, rejected

In [11]:
# Apply validation rules to PNU dataset

pnu_validated, pnu_rejected = validate_dataset(df)


print("PNU total:", len(df))
print("PNU validated:", len(pnu_validated))
print("PNU rejected:", len(pnu_rejected))

PNU total: 1544
PNU validated: 1544
PNU rejected: 0


In [8]:
# Show the reasons why records failed validation

pnu_rejected["validation_error"].value_counts()

Series([], Name: count, dtype: Int64)

In [12]:
# Display validation failure reasons if any records fail

pnu_rejected["validation_error"].value_counts()

Series([], Name: count, dtype: Int64)

In [13]:
# Remove validation helper column before final output

pnu_validated = pnu_validated.drop(
    columns="validation_error",
    errors="ignore"
)

print("Final columns:", len(pnu_validated.columns))

Final columns: 13


## Save Validation Results

The validated and rejected records are saved separately for the next pipeline stage.

In [16]:
pnu_validated.to_csv(
    interim_dir / "PNU_task3_validated.csv",
    index=False
)

pnu_rejected.to_csv(
    interim_dir / "PNU_task3_rejected.csv",
    index=False
)
print("Validation files saved successfully.")

Validation files saved successfully.


## Final Schema Check

This section verifies that the validated PNU dataset contains all required common schema columns before moving to the next pipeline stage.

In [15]:
# Define the final common schema columns

common_schema_columns = [
    "research_id",
    "university",
    "title",
    "authors",
    "publication_year",
    "publication_date",
    "abstract",
    "research_field",
    "tech_category",
    "journal",
    "doi",
    "url",
    "source"
]


# Check missing columns from the final dataset

missing_columns = [
    column
    for column in common_schema_columns
    if column not in pnu_validated.columns
]


# Display schema validation results

print("Expected columns:", len(common_schema_columns))
print("Available columns:", len(pnu_validated.columns))
print("Missing columns:", missing_columns)


# Check nullable fields with missing values

print("\nMissing values by column:")
print(
    pnu_validated[common_schema_columns]
    .isna()
    .sum()
)

Expected columns: 13
Available columns: 13
Missing columns: []

Missing values by column:
research_id            0
university             0
title                  0
authors                0
publication_year       0
publication_date     887
abstract             914
research_field         0
tech_category       1544
journal               13
doi                    0
url                    0
source                 0
dtype: int64


# Task 3 Summary
 
 The PNU validated dataset was checked against the project common schema to ensure consistency and readiness for integration with other university datasets.
 
 Validation steps included:
 - Verifying that all required common schema columns are available.
 - Checking mandatory fields based on the defined schema rules.
 - Validating publication year format.
 - Checking duplicate research IDs.
 - Validating URL format.
 - Separating validated and rejected records with validation reasons.
 
 The dataset passed schema validation successfully.
 
 All common schema columns are available, and optional fields remain nullable when information is unavailable.
 
 The final validated dataset was saved successfully for the next pipeline stage:
 - `data/interim/PNU_task3_validated.csv`
 - `data/interim/PNU_task3_rejected.csv`